In [ ]:
# ==============================================================================
# 1️⃣ 설정 및 라이브러리 임포트
# ==============================================================================
import os
import re
import warnings
import traceback
import numpy as np
import pandas as pd
from numba import njit
import quantstats as qs
from tqdm.auto import tqdm
from datetime import datetime
from zoneinfo import ZoneInfo
import google.generativeai as genai

from config import *

# 경고 무시
warnings.filterwarnings('ignore')

# [설정] 경로 및 API 키
type_ = "Momentum"
BASE_DIR = "/Users/cyberedjs/Desktop/Unity/data"
DATA_DIR = os.path.join(BASE_DIR, "combined") # combined data location
RESULT_DIR = f"/Users/cyberedjs/Desktop/Unity/results/pa_trade_results/{type_}"
STATUS_DIRS = ["pass", "fail", "hold", "priority"]

# 폴더 생성
os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(f"{RESULT_DIR}/pass", exist_ok=True)
os.makedirs(f"{RESULT_DIR}/hold", exist_ok=True)
os.makedirs(f"{RESULT_DIR}/fail", exist_ok=True)
os.makedirs(f"{RESULT_DIR}/priority", exist_ok=True)

# Gemini 설정
genai.configure(api_key=GOOGLE_API_KEY)

# ==============================================================================
# 2️⃣ 데이터 로더 (Data Loader)
# ==============================================================================
def load_market_data(data_dir):
    """
    Open, High, Low, Close, Volume 데이터를 로드하여 딕셔너리로 반환
    """
    print("[Data Loader] 데이터 로딩 시작...")
    try:
        data = {}
        for col in ['open', 'high', 'low', 'close', 'volume']:
            path = os.path.join(data_dir, f"{col}_15m.parquet")
            if os.path.exists(path):
                # 빠른 로딩을 위해 pyarrow 엔진 사용
                df = pd.read_parquet(path, engine='pyarrow')
                # 날짜 인덱스 보장
                if not isinstance(df.index, pd.DatetimeIndex):
                    df.index = pd.to_datetime(df.index)
                data[col] = df
            else:
                raise FileNotFoundError(f"{path} 파일이 없습니다.")
        
        print(f"[Data Loader] 로딩 완료. 데이터 크기: {data['close'].shape}")
        return data
    except Exception as e:
        print(f"[Error] 데이터 로딩 실패: {e}")
        return None

# ==============================================================================
# 3️⃣ AI Agent: PA Architect 
# ==============================================================================       
class PA_Architect_Agent:
    def __init__(self, model_name='gemini-2.5-flash'):
        self.model = genai.GenerativeModel(model_name)

    def search_and_generate_strategy(self, strategy_type="Trend Following", existing_strategies=None):
        """
        기존 전략명을 인식하고, 새로운 이름의 전략과 코드를 생성
        """
        print(f"[AI Agent] '{strategy_type}' 전략 검색 및 코드 생성 중...")

        excluded_strategies = ", ".join(existing_strategies) if existing_strategies else "None"

        prompt = f"""

        You are an expert Crypto Quant Researcher AND a high-performance Python engineer.

        You will generate ONE trading strategy including a generate_signals() function.
        You MUST NOT use numba or any Python lists inside the signal generation logic.
        The output must follow all of these rules strictly.

        ============================================================
        GOAL
        ============================================================
        Create ONE valid strategy of type: {strategy_type}

        The strategy MUST:
        - Work on 15-minute OHLCV crypto data
        - Be popular and widely used from trading communities, such as tradingview or youtube, etc.
        - Be different from these existing strategies:
        {excluded_strategies}
        - Have BOTH entry and exit logic clearly defined
        - Use only numpy vectorized operations (NO Python loops)

        ============================================================
        MANDATORY FORMAT
        ============================================================
        You must output EXACTLY this structure:

        Strategy Name: <NAME>
        Description:
        <150 words max>

        ------------------------------------------------------------
        ```python
        # FULL PYTHON CODE
        # MUST include generate_signals()

        ============================================================
        ABSOLUTE RULES FOR generate_signals()

        Function signature:
        def generate_signals(opens, highs, lows, closes, volumes):

        Inputs:
            •	opens, highs, lows, closes, volumes: pandas.DataFrame
            •	shape: (T, N)
            •	ALL columns are symbols
            •	index is DatetimeIndex

        Rules:
            1.	Inside generate_signals(), you MUST convert to numpy immediately:
        open_arr = opens.values.astype(np.float64)
        …
        No pandas operations allowed after conversion.
            2.	NO numba, NO cython, NO lists, NO append, NO dict, NO loops.
            3.	ALL logic MUST use numpy vectorized operations.
            4.	The function MUST return:
        pandas.DataFrame(signals, index=index, columns=columns, dtype=np.int8)
            5.	signals must be (T, N) with values in { -1, 0, 1 }
            6.	Signal meaning:
        1  = long position
        -1 = short position
        0  = flat / exit
        If signal[t-1] = 0 and signal[t] = 1 → enter long at close[t]
            7.	Strategy logic must apply per-column, vectorized across all symbols.

        ============================================================
        RESTRICTIONS INSIDE THE CODE BLOCK
            •	NO markdown
            •	NO backticks except the main ```python fenced block
            •	NO text outside the required format
            •	Only numpy + pandas allowed
            •	No loops over T or N (no for t in range(T), no for j in range(N))

        ============================================================
        OUTPUT REQUIREMENTS (STRICT)
            •	EXACTLY ONE code block
            •	Code block starts with ```python
            •	Ends with ```
            •	The rest must be plain text (Strategy name + Description)

        """

        try:
            response = self.model.generate_content(prompt)
            content = response.text

            # 전략 이름 추출
            name_match = re.search(r"Strategy Name:\s*(.+)", content)
            strategy_name = name_match.group(1).strip() if name_match else f"{strategy_type}_Unnamed"

            # 코드 블록 추출
            code_match = re.search(r"```python\n(.*?)```", content, re.DOTALL)
            code = code_match.group(1).strip() if code_match else None

            return strategy_name, content, code

        except Exception as e:
            print(f"[Error] AI 응답 처리 실패: {e}")
            return None, None, None
        
    def fix_code_with_error(self, code, error_log):
        """에러 로그를 기반으로 LLM에게 코드 수정 요청"""
        print("[AI Agent] 코드 수정 요청 중 (Error-driven Fix)...")
        prompt = f"""
        The following Python code raised an execution error.
        Please FIX the code based on the error message below.

        --- ERROR LOG ---
        {error_log}
        -----------------

        The original code:
        ```python
        {code}
        ```

        Rules:
        - Keep the same strategy logic as much as possible
        - Fix the bug causing the error
        """

        response = self.model.generate_content(prompt)
        fixed_code_match = re.search(r"```python\n(.*?)```", response.text, re.DOTALL)
        if fixed_code_match:
            return fixed_code_match.group(1).strip()
        else:
            return code  # fallback

    def generate_and_run(self, strategy_type, existing_strategies, data_dict, max_retries=3):

        strategy_name, desc, code = self.search_and_generate_strategy(strategy_type, existing_strategies)

        for attempt in range(max_retries):
            print(f"\n🚀 [Try {attempt+1}/{max_retries}] 전략 실행 중: {strategy_name}")
            try:
                trades_df = execute_strategy(data_dict, code)

                # 정상 실행 + 거래 발생
                if trades_df is not None and not trades_df.empty:
                    print("✅ 실행 성공! 거래 발생.")
                    return strategy_name, desc, code, trades_df

                # 정상 실행 + 거래 없음 (전략 조건이 너무 엄격함)
                elif trades_df is not None:
                    print("⚠️ 거래 없음 — 전략 조건이 너무 엄격합니다.")
                    # 👉 거래 없음일 때는 LLM이 조건을 완화해서 재시도하게 함
                    code = self.fix_code_with_error(code, "No trades detected — Something Wrong with the entry conditions or entry condition is too tight. Double check and fix the logic and error. \n NB: ABSOLUTELY NO explicit or implicit loops over rows or columns.")
                    continue

            except Exception as e:
                import traceback
                error_log = traceback.format_exc()
                print(f"[Error] 실행 실패: {str(e)}")
                print("[AI Agent] 오류 로그 기반 코드 수정 시도 중...")
                code = self.fix_code_with_error(code, error_log)
                continue  # 다음 시도

        print("❌ 모든 시도 실패 — 수동 검토 필요.")
        return strategy_name, desc, code, trades_df
    
# ==============================================================================
# 4️⃣ 백테스팅 엔진 (수정됨: 스코프 통합으로 NameError 해결)
# ==============================================================================

def execute_strategy(data_dict, strategy_code, manual=False, signals=None):
    """
    Backtesting wrapper:
      - executes strategy_code to obtain generate_signals (if not manual)
      - expects generate_signals to accept DataFrame (T,N) and return DataFrame signals (T,N)
      - uses ultra-fast event-driven backtester (vectorized)
    """
    print("[Backtest] 전략 코드 실행 및 거래 시뮬레이션...")
    execution_scope = {}

    if manual:
        if isinstance(signals, pd.DataFrame):
            signals_df = signals.fillna(0).astype(np.int8)
        else:
            raise ValueError("manual=True일 때 signals는 pandas.DataFrame이어야 합니다.")
    else:
        try:
            execution_scope = {'np': np, 'pd': pd}
            exec(strategy_code, execution_scope)
            
            if 'generate_signals' not in execution_scope:
                raise ValueError("generate_signals 함수가 코드에 없습니다.")
            generate_signals = execution_scope['generate_signals']

            signals_df = generate_signals(
                data_dict['open'], data_dict['high'], data_dict['low'],
                data_dict['close'], data_dict['volume']
            )
            if not isinstance(signals_df, pd.DataFrame):
                if isinstance(signals_df, np.ndarray) and signals_df.ndim == 2:
                    signals_df = pd.DataFrame(
                        signals_df,
                        index=data_dict['close'].index,
                        columns=data_dict['close'].columns
                    )
                else:
                    raise ValueError("generate_signals 반환값이 DataFrame 또는 2D numpy array가 아닙니다.")

            signals_df = signals_df.fillna(0).astype(np.int8)

        except Exception:
            import traceback
            print("[Error] 전략 코드 실행 중 예외 발생:\n" + traceback.format_exc())
            raise

    # ================================
    # Ultra-fast backtest START
    # ================================
    print("[Backtest] 초고속 벡터 기반 백테스트 실행 중...")

    closes = data_dict['close']
    T, N = closes.shape

    sig = signals_df.values.astype(np.int8)
    close_arr = closes.values.astype(np.float64)

    index = closes.index
    columns = closes.columns

    # ---- 1. Entry/Exit event detection ----
    prev = np.zeros_like(sig)
    prev[1:] = sig[:-1]

    entry_mask = (prev == 0) & (sig != 0)
    exit_mask  = (prev != 0) & (sig == 0)

    trades = []  # list of dicts

    for j in range(N):
        sym = columns[j]

        entry_idx = np.nonzero(entry_mask[:, j])[0]
        exit_idx  = np.nonzero(exit_mask[:, j])[0]

        m = min(len(entry_idx), len(exit_idx))
        if m == 0:
            continue

        for k in range(m):
            e = int(entry_idx[k])
            x = int(exit_idx[k])
            if x <= e:
                continue

            direction = int(sig[e, j])
            entry_price = float(close_arr[e, j])
            exit_price  = float(close_arr[x, j])

            # long/short returns
            if direction == 1:
                ret = (exit_price - entry_price) / entry_price
            else:
                ret = (entry_price - exit_price) / entry_price

            # no fee here, but can add
            trades.append({
                'symbol': sym,
                'entry_time': index[e],
                'exit_time': index[x],
                'entry_price': entry_price,
                'exit_price': exit_price,
                'direction': direction,
                'return': float(ret),
                'run_up': np.nan,    # old numba values not computed
                'run_down': np.nan
            })

    # ---- Final return ----
    if len(trades) == 0:
        return pd.DataFrame(columns=[
            'symbol','entry_time','exit_time','entry_price','exit_price',
            'direction','return','run_up','run_down'
        ])

    trades_df = pd.DataFrame(trades)
    trades_df['entry_time'] = pd.to_datetime(trades_df['entry_time'])
    trades_df['exit_time']  = pd.to_datetime(trades_df['exit_time'])
    return trades_df

# ==============================================================================
# 5️⃣ 메인 실행 함수 (수정됨)
# ==============================================================================
def main():

    # 전략 상태 폴더들    
    existing_strategies = set()  # 중복 방지

    for status in STATUS_DIRS:
        status_dir = os.path.join(RESULT_DIR, status)
        if not os.path.exists(status_dir):
            continue

        for name in os.listdir(status_dir):
            path = os.path.join(status_dir, name)
            if os.path.isdir(path):
                existing_strategies.add(name)

    existing_strategies = list(existing_strategies)
    print(f"[Info] 현재 저장된 전략의 개수: {len(existing_strategies)}개")

    # 2. 데이터 로드
    market_data = load_market_data(DATA_DIR)
    if market_data is None:
        print("[Error] 데이터 로딩 실패. 프로그램 종료.")
        return

    architect = PA_Architect_Agent()

    # 3️⃣ 무한 루프 — 새로운 전략을 계속 생성
    while True:
        print("\n" + "="*80)
        print(f"[RUNNING] 새로운 전략 생성 및 백테스트 시도 중... (현재 {len(existing_strategies)}개 저장됨)")
        print("="*80 + "\n")

        try:
            
            # 🔥 단일 스레드로 바로 실행
            strategy_name, strategy_doc, strategy_code, trades_df = \
                architect.generate_and_run(
                    type_,
                    existing_strategies,
                    market_data
                )

            # 실패 또는 거래 없음
            if trades_df is None or trades_df.empty:
                print(f"⚠️ 전략 '{strategy_name}' 거래 없음 또는 실패 — 다음 전략으로 이동")
                if strategy_name:
                    existing_strategies.append(strategy_name)
                continue

            # 성공 시 저장
            print("\n" + "="*50)
            print(f"🤖 생성된 전략명: {strategy_name}")
            print("="*50 + "\n")

            # 새로운 전략 저장 경로
            NEW_STRATEGY_BASE_DIR = os.path.join(RESULT_DIR, "hold")
            os.makedirs(NEW_STRATEGY_BASE_DIR, exist_ok=True)

            # 예시: 새로운 전략 이름
            new_strategy_name = strategy_name  # 또는 strategy_name

            new_strategy_dir = os.path.join(NEW_STRATEGY_BASE_DIR, new_strategy_name)
            os.makedirs(new_strategy_dir, exist_ok=True)

            # (1) 전략 설명 저장
            desc_path = os.path.join(new_strategy_dir, f"Strategy_Description.txt")
            with open(desc_path, "w", encoding="utf-8") as f:
                f.write(strategy_doc)
            print(f"[Save] 전략 설명 저장 완료: {desc_path}")

            # (2) 전체 거래 내역 저장
            trades_df['anchor'] = trades_df['entry_time'].dt.floor('H')
            all_csv_path = os.path.join(new_strategy_dir, f"PA_All_Trades.csv")
            trades_df.to_csv(all_csv_path, index=False)
            print(f"[Save] 전체 거래 내역 저장 완료 ({len(trades_df)}건): {all_csv_path}")

            # (3) anchor별 TOP3 거래 내역 저장
            top3_df = trades_df.groupby('anchor').apply(
                lambda x: x.nlargest(3, 'return')
            ).reset_index(drop=True)
            top3_csv_path = os.path.join(new_strategy_dir, f"PA_Top3_Trades.csv")
            top3_df.to_csv(top3_csv_path, index=False)
            print(f"[Save] TOP3 거래 내역 저장 완료: {top3_csv_path}")

            # (4) QuantStats 리포트 생성
            try:
                aum = (top3_df.set_index('exit_time').sort_index()['return'] * 1000).cumsum() + 10000
                aum_daily = aum.resample('D').last().ffill()

                html_path = os.path.join(new_strategy_dir, f"PA_Report.html")
                qs.reports.html(aum_daily.pct_change(), output=html_path, title=f"{strategy_name} Report")
                print(f"[Save] 리포트 생성 완료: {html_path}")
            except Exception as e:
                print(f"[Warning] HTML 리포트 생성 중 오류 발생: {e}")

            # 성공적으로 저장된 전략을 목록에 추가
            existing_strategies.append(strategy_name)

        except Exception as e:
            import traceback
            error_log = traceback.format_exc()
            print(f"[Critical Error] 예기치 못한 예외 발생:\n{error_log}")
            print("➡️ 해당 전략 건너뛰고 다음으로 진행합니다.")
            # 그래도 실패한 전략 이름이 있다면 추가
            if 'strategy_name' in locals():
                existing_strategies.append(strategy_name)
            continue

if __name__ == "__main__":
    # 실행
    main()